# RUDDER microscopy — Kalman correction (train from scratch)

**Independent Google Colab notebook.** Loads the same original EELS file using `gdown`, prepares the measured image and spectral targets, and trains a new GRU return predictor and PPO policy; **no external project files or pretrained weights**.

**Correction:** Kalman filter forecasts drift to the midpoint of each spectrum; calibration observes noisy drift.

The two notebooks use the same data file, split seed, action space, budget, drift processes, training seeds, and evaluation seeds. Results must be compared on the same configuration. **Warning:** the `gdown` source is ~584 MB; a public downloadable Google Drive link is required. The raw source has not been downloaded or executed in this creation environment; Colab is the intended execution environment.

The scientific return is $G=U_0-U_T$; intermediate RUDDER credits come from GRU prediction increments plus a terminal residual. RUDDER and PPO are trained **inside this notebook**.


In [ ]:
#@title Standard imports and experiment controls
import os, sys, io, hashlib, json, copy, time, unittest
from dataclasses import dataclass, field, asdict, replace
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import numpy as np
import scipy
from scipy.ndimage import gaussian_filter1d, map_coordinates
from scipy.spatial import cKDTree
import sklearn
from sklearn.ensemble import RandomForestRegressor
import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(1)

# FULL runs train the RUDDER predictor and PPO from scratch; edit counts for smoke test.
# 64 + 384 campaigns per branch/seed are used by default.
RUN_TRAINING = True
EVALUATION_SEEDS_PER_REGIME = 20
EVALUATION_FIRST_SEED = 950000
TRAINING_EPISODES = 384
WARMUP_EPISODES = 64
POLICY_BATCH_SIZE = 8
TRAINING_SEEDS = [0, 1, 2]
COST_RANDOMIZATION = False
DRIFT_REGIMES = ("random", "linear", "periodic")
# Quick diagnostic example: WARMUP_EPISODES=8; TRAINING_EPISODES=16; TRAINING_SEEDS=[0].

print("Current versions:", {"python":sys.version.split()[0], "numpy":np.__version__,
      "scipy":scipy.__version__, "scikit-learn":sklearn.__version__, "torch":torch.__version__})


## Environment: image blur, time, spectra, drift, and scientific random forest
The simulator uses eight IMAGE settings, CALIBRATE, and SPECTRUM. The finite measured field defines valid paired spectra.

In [ ]:
#@title Full scientific learner and base environment
"""Budgeted microscopy with real EELS measurements and simulated drift/blur.

Policy input is ``observation()`` only. True drift, physical positions, nearest
pixel errors and hidden test labels occur only in simulator/audit objects.
Calibration is an ideal reactive registration measurement at its completion:
it returns the beam to the material anchor of the cached image and leaves that
image intact. No future drift prediction or Kalman filter is present.

Finite-field limitation: image patches use reflected padding, matching the
source notebook, while spectroscopy chooses the nearest eligible measured
pixel only for centers inside the measured field. An outside-field spectroscopy
attempt consumes time but produces no supervised pair. Those failure rates and
nearest-pixel distances are explicitly audited.
"""
from __future__ import annotations
from dataclasses import dataclass, field
import hashlib
import json
from pathlib import Path
from typing import Any
import numpy as np
from scipy.ndimage import gaussian_filter1d, map_coordinates
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
IMAGE_ACTIONS = tuple(range(8))
CALIBRATE = 8
SPECTRUM = 9
N_ACTIONS = 10
PATCH_SIZE = 17
POOL_SIZE = 7
_GRID_R, _GRID_C = np.meshgrid(np.arange(-8, 9), np.arange(-8, 9), indexing='ij')
_POOL_WEIGHTS = np.zeros((POOL_SIZE, PATCH_SIZE), np.float32)
for _i in range(POOL_SIZE):
    _start, _end = (int(np.floor(_i * PATCH_SIZE / POOL_SIZE)), int(np.ceil((_i + 1) * PATCH_SIZE / POOL_SIZE)))
    _POOL_WEIGHTS[_i, _start:_end] = 1.0 / (_end - _start)

@dataclass(frozen=True)
class BaseConfig:
    budget: float = 65.0
    taus: tuple[float, ...] = tuple((float(x) for x in np.linspace(0.2, 1.0, 8)))
    image_cost_factor: float = 3.0
    cal_cost: float = 1.0
    spectrum_duration: float = 1.0
    survey_cost: float = 5.0
    seed_tau: float = 0.8
    seed_pairs: int = 4
    ref_size: int = 64
    rf_trees: int = 24
    rf_min_samples_leaf: int = 2
    rf_max_features: float = 0.8
    rf_seed: int = 42
    max_images_per_target: int | None = None
    velocity: float = 0.4
    drift_period: float = 48.0
    random_walk_scale: float = 0.16
    periodic_amplitude: float = 3.0
    regimes: tuple[str, ...] = ('random', 'linear', 'periodic')

    def __post_init__(self) -> None:
        if len(self.taus) != 8 or any((not 0 < x <= 1 for x in self.taus)):
            raise ValueError('Exactly eight positive acquisition fractions <=1 are required.')
        if min(self.budget, self.image_cost_factor, self.cal_cost, self.spectrum_duration) <= 0:
            raise ValueError('Budget and acquisition durations must be positive.')
        if self.seed_pairs < 4 or self.ref_size < 1 or self.rf_trees < 2:
            raise ValueError('Insufficient initialization/model size or invalid image limit.')
        if self.max_images_per_target is not None and self.max_images_per_target < 1:
            raise ValueError('Optional image limit must be positive.')

def blur_sigma(tau: float) -> float:
    return float(np.sqrt(max(0.0, (1.0 / float(tau) - 1.0) / 0.12)))

def pool_features(patches: np.ndarray) -> np.ndarray:
    """Equivalent to a 17x17 -> 7x7 adaptive average pool; no Torch dependency."""
    arr = np.asarray(patches, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[None]
    if arr.shape[-2:] != (PATCH_SIZE, PATCH_SIZE):
        raise ValueError('Expected one or more 17x17 structural patches.')
    pooled = np.matmul(_POOL_WEIGHTS, np.matmul(arr, _POOL_WEIGHTS.T))
    return pooled.reshape(len(arr), -1)

@dataclass
class DataBundle:
    image: np.ndarray
    pool_ids: np.ndarray
    seed_ids: np.ndarray
    ref_ids: np.ndarray
    test_ids: np.ndarray
    pool_targets: np.ndarray
    seed_targets: np.ndarray
    hidden_test_targets: np.ndarray
    metadata: dict[str, Any]
    centers: np.ndarray = field(init=False)
    pool_tree: cKDTree = field(init=False, repr=False)
    image_mean: float = field(init=False)
    image_std: float = field(init=False)
    _feature_cache: dict = field(default_factory=dict, init=False, repr=False)

    def __post_init__(self) -> None:
        h, w = self.image.shape
        self.centers = np.stack(np.unravel_index(np.arange(h * w), (h, w)), axis=1).astype(np.float64)
        self.pool_tree = cKDTree(self.centers[self.pool_ids])
        self.image_mean = float(self.image.mean())
        self.image_std = max(float(self.image.std()), 1e-08)
        groups = [self.pool_ids, self.seed_ids, self.ref_ids, self.test_ids]
        for i, left in enumerate(groups):
            for right in groups[i + 1:]:
                if np.intersect1d(left, right).size:
                    raise ValueError('Learning/reference/test partitions overlap.')
        if self.pool_targets.shape != (len(self.pool_ids), 5) or self.seed_targets.shape != (len(self.seed_ids), 5):
            raise ValueError('Measured target shape mismatch.')
        if self.hidden_test_targets.shape != (len(self.test_ids), 5):
            raise ValueError('Hidden test target shape mismatch.')

    def crop(self, position: np.ndarray, tau: float=1.0) -> np.ndarray:
        row, col = np.asarray(position, dtype=float)
        patch = map_coordinates(self.image, [row + _GRID_R, col + _GRID_C], order=1, mode='reflect')
        sigma = blur_sigma(tau)
        if sigma > 1e-08:
            patch = gaussian_filter1d(patch, sigma, axis=1, mode='reflect')
        return patch.astype(np.float32)

    def fixed_features(self, ids: np.ndarray, tau: float=1.0) -> np.ndarray:
        ids = np.asarray(ids, dtype=np.int64)
        key = (ids.tobytes(), float(tau))
        if key not in self._feature_cache:
            self._feature_cache[key] = pool_features(np.stack([self.crop(self.centers[i], tau) for i in ids]))
        return self._feature_cache[key]

class DriftWorld:
    """Continuous queryable latent trajectory, fixed for a complete campaign."""

    def __init__(self, regime: str, seed: int, cfg: BaseConfig):
        if regime not in {'random', 'linear', 'periodic'}:
            raise ValueError(f'Unknown drift regime {regime!r}')
        self.regime, self.seed, self.cfg = (regime, int(seed), cfg)
        rng = np.random.default_rng(seed)
        angle = rng.uniform(0, 2 * np.pi)
        self.velocity = cfg.velocity * np.array([np.cos(angle), np.sin(angle)])
        self.phase = rng.uniform(0, 2 * np.pi)
        count = max(2000, int(np.ceil(cfg.budget)) + 4)
        increments = np.random.default_rng(seed + 100000).normal(scale=cfg.random_walk_scale, size=(count, 2))
        self.knots = np.vstack([np.zeros((1, 2)), np.cumsum(increments, axis=0)])

    def true(self, time: float) -> np.ndarray:
        time = max(0.0, float(time))
        index = min(int(time), len(self.knots) - 2)
        fraction = time - index
        random_walk = (1 - fraction) * self.knots[index] + fraction * self.knots[index + 1]
        if self.regime == 'random':
            return random_walk
        linear = self.velocity * time + 0.18 * random_walk
        if self.regime == 'linear':
            return linear
        theta = 2 * np.pi * time / self.cfg.drift_period + self.phase
        periodic = np.array([np.sin(theta) - np.sin(self.phase), np.cos(theta) - np.cos(self.phase)])
        return linear + self.cfg.periodic_amplitude * periodic

class ScienceLearner:
    """Same RF and target representation for every controller/reward method."""

    def __init__(self, data: DataBundle, cfg: BaseConfig, initial_indices: np.ndarray):
        self.data, self.cfg = (data, cfg)
        self.initial_ids = data.seed_ids[initial_indices].copy()
        self.train_features = [row.copy() for row in data.fixed_features(self.initial_ids, cfg.seed_tau)]
        self.train_targets = [row.copy() for row in data.seed_targets[initial_indices]]
        self.measured_ids = [int(x) for x in self.initial_ids]
        self.reference_ids = data.ref_ids[:min(cfg.ref_size, len(data.ref_ids))].copy()
        self.reference_features = data.fixed_features(self.reference_ids, 1.0)
        self.n_updates = 0
        self.refit()

    def refit(self) -> None:
        self.model = RandomForestRegressor(n_estimators=self.cfg.rf_trees, min_samples_leaf=self.cfg.rf_min_samples_leaf, max_features=self.cfg.rf_max_features, n_jobs=1, random_state=self.cfg.rf_seed)
        self.model.fit(np.stack(self.train_features), np.stack(self.train_targets))
        individual = np.stack([tree.predict(self.reference_features) for tree in self.model.estimators_])
        self.current_U = float(individual.var(axis=0, ddof=0).mean())

    def append(self, patch: np.ndarray, pool_index: int) -> float:
        old_u = self.current_U
        self.train_features.append(pool_features(patch)[0])
        self.train_targets.append(self.data.pool_targets[pool_index].copy())
        self.measured_ids.append(int(self.data.pool_ids[pool_index]))
        self.n_updates += 1
        self.refit()
        return float(old_u - self.current_U)

    def hidden_test_mse(self) -> float:
        features = self.data.fixed_features(self.data.test_ids, 1.0)
        prediction = self.model.predict(features)
        return float(np.mean((prediction - self.data.hidden_test_targets) ** 2))

class BaseCampaignEnv:
    """One episode is one complete paid campaign; actions 0..7 IMAGE,8 CAL,9 SPEC."""
    obs_names = ('time_remaining', 'has_cached_image', 'image_calibrated', 'last_tau', 'image_age', 'successful_pairs_fraction', 'images_fraction', 'calibrations_fraction', 'relative_uncertainty', 'last_calibration_age', 'last_measured_shift_row', 'last_measured_shift_col', 'commanded_correction_row', 'commanded_correction_col', 'images_this_target', 'time_since_spectrum', 'image_gradient', 'image_contrast', 'initial_uncertainty', 'current_uncertainty') + tuple((f'image_pool_{i}' for i in range(49)))

    def __init__(self, data: DataBundle, cfg: BaseConfig | None=None, seed: int=0, regime: str | None=None):
        self.data, self.cfg, self.seed = (data, cfg or BaseConfig(), int(seed))
        self.regime = regime or self.cfg.regimes[self.seed % len(self.cfg.regimes)]
        self.reset()

    def reset(self) -> np.ndarray:
        cfg = self.cfg
        self.world = DriftWorld(self.regime, self.seed + 1500, cfg)
        self.target_order = np.random.default_rng(self.seed + 700).permutation(self.data.pool_ids)
        self.cursor = 0
        self.target_id = int(self.target_order[0])
        initial_indices = np.random.default_rng(self.seed + 900).choice(len(self.data.seed_ids), size=min(cfg.seed_pairs, len(self.data.seed_ids)), replace=False)
        self.science = ScienceLearner(self.data, cfg, initial_indices)
        self.initial_cost = cfg.survey_cost + len(initial_indices) * (cfg.image_cost_factor * cfg.seed_tau + cfg.spectrum_duration)
        self.t = float(self.initial_cost)
        if self.t + cfg.image_cost_factor * min(cfg.taus) + cfg.spectrum_duration > cfg.budget:
            raise ValueError('Budget cannot cover the common survey, registered seed pairs, and one measurement.')
        self.U0 = float(self.science.current_U)
        self.correction = self.world.true(self.t).copy()
        self.last_measured_shift = self.correction.copy()
        self.last_calibration_time = self.t
        self.initial_heldout_mse = self.science.hidden_test_mse()
        self.last_spectrum_time = self.t
        self.cached_image = None
        self.cached_anchor = None
        self.last_tau = None
        self.image_midpoint = None
        self.calibrated = False
        self.images_this_target = 0
        self.counts = {'image': 0, 'calibrate': 0, 'spectrum': 0, 'successful_pairs': 0, 'failed_spectra': 0}
        self.image_action_counts = np.zeros(8, dtype=int)
        self.trace: list[dict[str, Any]] = []
        self.spec_trace: list[dict[str, Any]] = []
        self.done = False
        return self.observation()

    @property
    def terminal_score(self) -> float:
        return float(self.U0 - self.science.current_U)

    @property
    def Ufinal(self) -> float:
        return float(self.science.current_U)

    def action_mask(self) -> np.ndarray:
        mask = np.zeros(N_ACTIONS, dtype=bool)
        if self.done:
            return mask
        cfg = self.cfg
        if cfg.max_images_per_target is None or self.images_this_target < cfg.max_images_per_target:
            mask[:8] = self.t + cfg.image_cost_factor * np.asarray(cfg.taus) + cfg.spectrum_duration <= cfg.budget + 1e-09
        if self.cached_image is not None:
            mask[CALIBRATE] = self.t + cfg.cal_cost + cfg.spectrum_duration <= cfg.budget + 1e-09
            mask[SPECTRUM] = self.t + cfg.spectrum_duration <= cfg.budget + 1e-09
        return mask

    def observation(self) -> np.ndarray:
        cfg = self.cfg
        cached = self.cached_image is not None
        if cached:
            normalized = (self.cached_image - self.data.image_mean) / self.data.image_std
            feature = pool_features(normalized)[0]
            gradient = float(np.mean(np.abs(np.diff(normalized, axis=1))))
            contrast = float(normalized.std())
        else:
            feature = np.zeros(49, dtype=np.float32)
            gradient = contrast = 0.0
        cal_age = 1.0 if self.last_calibration_time is None else (self.t - self.last_calibration_time) / cfg.budget
        scalar = [max(0.0, (cfg.budget - self.t) / cfg.budget), float(cached), float(self.calibrated), float(self.last_tau or 0.0), (self.t - self.image_midpoint) / cfg.budget if cached else 0.0, self.counts['successful_pairs'] / cfg.budget, self.counts['image'] / cfg.budget, self.counts['calibrate'] / cfg.budget, min(10.0, self.science.current_U / max(self.U0, 1e-06)), cal_age, *self.last_measured_shift / PATCH_SIZE, *self.correction / PATCH_SIZE, min(1.0, self.images_this_target / (cfg.budget / (cfg.image_cost_factor * min(cfg.taus)))), (self.t - self.last_spectrum_time) / cfg.budget, gradient, contrast, self.U0, self.science.current_U]
        result = np.concatenate([np.asarray(scalar, np.float32), feature]).astype(np.float32)
        if result.shape != (len(BaseCampaignEnv.obs_names),) or not np.isfinite(result).all():
            raise RuntimeError('Invalid policy observation.')
        return result

    def step(self, action: int) -> tuple[np.ndarray, float, bool, dict[str, Any]]:
        action = int(action)
        if not 0 <= action < N_ACTIONS or not self.action_mask()[action]:
            raise ValueError(f'Illegal action {action} at time {self.t:.6f}')
        cfg, start = (self.cfg, self.t)
        target_id = self.target_id
        target = self.data.centers[target_id]
        old_u = self.science.current_U
        event: dict[str, Any] = {'step': len(self.trace), 'action': action, 'time_start': float(start), 'target_id': int(target_id), 'U_before': float(old_u)}
        if action in IMAGE_ACTIONS:
            tau = cfg.taus[action]
            duration = cfg.image_cost_factor * tau
            midpoint = start + duration / 2
            position = target + self.world.true(midpoint) - self.correction
            self.cached_image = self.data.crop(position, tau)
            self.cached_anchor = position.copy()
            self.image_midpoint = midpoint
            self.last_tau = tau
            self.calibrated = False
            self.images_this_target += 1
            self.counts['image'] += 1
            self.image_action_counts[action] += 1
            self.t = start + duration
            h, w = self.data.image.shape
            outside = bool(position[0] < 0 or position[0] > h - 1 or position[1] < 0 or (position[1] > w - 1))
            padded = bool(position[0] < 8 or position[0] > h - 9 or position[1] < 8 or (position[1] > w - 9))
            event.update(kind='IMAGE', tau=float(tau), blur_sigma=blur_sigma(tau), sample_time=float(midpoint), image_midpoint=float(midpoint), actual_position=position.tolist(), image_anchor=position.tolist(), outside_source_field=outside, patch_uses_reflected_padding=padded)
        elif action == CALIBRATE:
            duration = cfg.cal_cost
            self.t = start + duration
            previous = self.correction.copy()
            self.correction = target + self.world.true(self.t) - self.cached_anchor
            self.last_measured_shift = self.correction - previous
            self.last_calibration_time = self.t
            self.calibrated = True
            self.counts['calibrate'] += 1
            event.update(kind='CALIBRATE', measured_shift=self.last_measured_shift.tolist(), correction=self.correction.tolist(), sample_time=float(self.t), actual_position=(target + self.world.true(self.t) - self.correction).tolist(), image_anchor=self.cached_anchor.tolist())
        else:
            duration = cfg.spectrum_duration
            midpoint = start + duration / 2
            position = target + self.world.true(midpoint) - self.correction
            self.t = start + duration
            h, w = self.data.image.shape
            outside = bool(position[0] < 0 or position[0] > h - 1 or position[1] < 0 or (position[1] > w - 1))
            quantization = real_id = source_position = measured_mismatch = None
            if not outside:
                quantization, pool_index = self.data.pool_tree.query(position)
                pool_index = int(pool_index)
                real_id = int(self.data.pool_ids[pool_index])
                source_position = self.data.centers[real_id].tolist()
                measured_mismatch = float(np.linalg.norm(self.data.centers[real_id] - self.cached_anchor))
                self.science.append(self.cached_image, pool_index)
                self.counts['successful_pairs'] += 1
            else:
                self.counts['failed_spectra'] += 1
            record = {'time': float(self.t), 'target_id': int(target_id), 'real_id': real_id, 'sample_time': float(midpoint), 'actual_position': position.tolist(), 'spectrum_midpoint': float(midpoint), 'spectrum_position': position.tolist(), 'source_id': real_id, 'source_position': source_position, 'image_anchor': self.cached_anchor.tolist(), 'image_midpoint': float(self.image_midpoint), 'pair_mismatch': float(np.linalg.norm(position - self.cached_anchor)), 'measured_pair_mismatch': measured_mismatch, 'absolute_target_error': float(np.linalg.norm(position - target)), 'nearest_spectrum_distance': float(quantization) if quantization is not None else None, 'quantization_error': float(quantization) if quantization is not None else None, 'outside_source_field': outside, 'successful_pair': not outside, 'acquisition_failed': outside, 'no_pair': outside, 'calibrated': bool(self.calibrated), 'tau': float(self.last_tau), 'n_images': int(self.images_this_target), 'U': self.science.current_U}
            self.spec_trace.append(record)
            event.update(kind='SPECTRUM', **{k: v for k, v in record.items() if k != 'target_id'})
            self.counts['spectrum'] += 1
            self.last_spectrum_time = self.t
            self.cursor += 1
            if self.cursor >= len(self.target_order):
                self.done = True
            else:
                self.target_id = int(self.target_order[self.cursor])
            self.cached_image = self.cached_anchor = None
            self.last_tau = self.image_midpoint = None
            self.calibrated = False
            self.images_this_target = 0
        if not self.action_mask().any():
            self.done = True
        if self.done:
            self.science.refit()
        reward = float(old_u - self.science.current_U)
        event.update(time_end=float(self.t), duration=float(duration), delta_U=reward, U_after=self.science.current_U, terminal=self.done)
        self.trace.append(event)
        if self.t > cfg.budget + 1e-08:
            raise RuntimeError('Campaign exceeded its experimental time budget.')
        info = {'action_name': event['kind'], 'duration': float(duration), 'delta_U': reward, 'time_used': self.t, 'terminal_score': self.terminal_score if self.done else None, 'audit': event.copy()}
        return (self.observation(), reward, self.done, info)

    def metrics(self) -> dict[str, Any]:

        def average(key: str) -> float:
            values = [x[key] for x in self.spec_trace if x[key] is not None]
            return float(np.mean(values)) if values else float('nan')
        images = [event for event in self.trace if event['kind'] == 'IMAGE']
        return {'seed': self.seed, 'regime': self.regime, 'budget': self.cfg.budget, 'time_used': float(self.t), 'initial_cost': float(self.initial_cost), 'U0': self.U0, 'Ufinal': self.Ufinal, 'terminal_score': self.terminal_score, 'heldout_mse': self.science.hidden_test_mse(), 'initial_heldout_mse': self.initial_heldout_mse, 'n_images': self.counts['image'], 'n_calibrations': self.counts['calibrate'], 'n_spectra': self.counts['spectrum'], 'n_seed_pairs': len(self.science.initial_ids), 'n_successful_pairs': self.counts['successful_pairs'], 'n_failed_spectra': self.counts['failed_spectra'], 'n_reference': len(self.science.reference_ids), 'counts': self.counts.copy(), 'image_action_counts': self.image_action_counts.tolist(), 'mean_pair_mismatch': average('pair_mismatch'), 'mean_measured_pair_mismatch': average('measured_pair_mismatch'), 'mean_absolute_target_error': average('absolute_target_error'), 'mean_nearest_spectrum_distance': average('nearest_spectrum_distance'), 'fraction_spectra_outside_source_field': average('outside_source_field'), 'fraction_images_outside_source_field': float(np.mean([x['outside_source_field'] for x in images])) if images else 0.0, 'fraction_image_patches_using_reflected_padding': float(np.mean([x['patch_uses_reflected_padding'] for x in images])) if images else 0.0, 'boundary_model': 'reflected image patches; outside-field SPEC costs time and yields no pair; in-field SPEC uses nearest eligible measured spectrum', 'setup_registration': 'common paid survey registers the initial map at t=initial_cost; drift clock is not reset', 'unique_acquired_spectra': len({x['real_id'] for x in self.spec_trace if x['real_id'] is not None})}

## Kalman filter and predictive spectrum positioning
A causal position–velocity filter is updated by noisy calibration observations and predicts the required correction to the spectrum midpoint.

In [ ]:
#@title Full causal Kalman filter
"""Causal Kalman estimation of the correction needed for a cached image.

The state is [required_correction_row, required_correction_col,
              drift_velocity_row, drift_velocity_col].
Its position component is NOT an estimate supplied with absolute drift or the
cached image's true material coordinates. At an image's midpoint the required
correction is, by definition, the known correction used to acquire that image.
Reanchoring there sets the position covariance and position/velocity cross
covariance to zero, while retaining the propagated velocity belief.

A registration sensor reports image/beam mismatch m. Its observable measurement
is z = currently_applied_correction + m. Only this z, its observation time and
measurement covariance enter observe(). Future spectrum commands come from the
nonmutating forecast(spectrum_midpoint), never from simulator truth.
"""
from __future__ import annotations
from dataclasses import dataclass
from typing import Any
import numpy as np

@dataclass(frozen=True)
class KalmanConfig:
    """Continuous process noise parameters in the simulator's physical units.

    q_position: direct position diffusion, pixels**2 / time.
    q_acceleration: white acceleration spectral density, pixels**2 / time**3.
    initial_velocity_std: per-axis prior standard deviation, pixels / time.
    """
    q_position: float = 0.0256
    q_acceleration: float = 0.0025
    initial_velocity_std: float = 0.4

    def __post_init__(self) -> None:
        for name in ('q_position', 'q_acceleration', 'initial_velocity_std'):
            value = float(getattr(self, name))
            if not np.isfinite(value) or value < 0:
                raise ValueError(f'{name} must be finite and nonnegative')

def _vector(value, size: int, name: str) -> np.ndarray:
    result = np.asarray(value, dtype=np.float64)
    if result.shape != (size,) or not np.isfinite(result).all():
        raise ValueError(f'{name} must be a finite vector of length {size}')
    return result.copy()

def _covariance(value, size: int, name: str) -> np.ndarray:
    result = np.asarray(value, dtype=np.float64)
    if result.shape != (size, size) or not np.isfinite(result).all():
        raise ValueError(f'{name} must be a finite {size}x{size} covariance')
    scale = max(1.0, float(np.max(np.abs(result))))
    if not np.allclose(result, result.T, rtol=1e-10, atol=1e-12 * scale):
        raise ValueError(f'{name} must be symmetric')
    result = (result + result.T) / 2
    eigenvalues = np.linalg.eigvalsh(result)
    if eigenvalues[0] < -1e-10 * scale:
        raise ValueError(f'{name} is not positive semidefinite')
    if eigenvalues[0] < 0:
        eigenvalues, eigenvectors = np.linalg.eigh(result)
        result = eigenvectors * np.maximum(eigenvalues, 0) @ eigenvectors.T
        result = (result + result.T) / 2
    return result.copy()

def transition_matrices(dt: float, q_position: float=0.0256, q_acceleration: float=0.0025) -> tuple[np.ndarray, np.ndarray]:
    """Exact discretization of position diffusion plus white acceleration.

    F = [[I, dt I], [0, I]]
    Q = [[(q_p dt + q_a dt**3/3) I, q_a dt**2/2 I],
         [q_a dt**2/2 I,                 q_a dt I]].
    """
    dt, qp, qa = (float(dt), float(q_position), float(q_acceleration))
    if not np.isfinite([dt, qp, qa]).all() or min(dt, qp, qa) < 0:
        raise ValueError('Time increments and process densities must be finite and nonnegative')
    eye = np.eye(2)
    F = np.eye(4)
    F[:2, 2:] = dt * eye
    Q = np.zeros((4, 4))
    Q[:2, :2] = (qp * dt + qa * dt ** 3 / 3) * eye
    Q[:2, 2:] = qa * dt ** 2 / 2 * eye
    Q[2:, :2] = Q[:2, 2:]
    Q[2:, 2:] = qa * dt * eye
    return (F, Q)

class KalmanDriftFilter:
    """Small, independent filter with absolute timestamps and causal updates.

    Constructor position is a known correction in the current reference frame.
    The default covariance treats it as known and supplies uncertainty only in
    velocity. An IMAGE action must call reanchor(applied_correction, midpoint)
    before propagating to its completion. No delayed observations are silently
    accepted; callers must process IMAGE midpoint/reference changes in order.

    Public x/P/time are convenient for the simulator, while all returned arrays
    are copies. forecast(time) never advances the stored posterior.
    """

    def __init__(self, position=None, velocity=None, *, time: float=0.0, covariance=None, config: KalmanConfig | None=None) -> None:
        self.config = config or KalmanConfig()
        self.time = float(time)
        if not np.isfinite(self.time) or self.time < 0:
            raise ValueError('Initial time must be finite and nonnegative')
        pos = _vector(np.zeros(2) if position is None else position, 2, 'position')
        vel = _vector(np.zeros(2) if velocity is None else velocity, 2, 'velocity')
        self.x = np.concatenate((pos, vel))
        if covariance is None:
            self.P = np.zeros((4, 4))
            self.P[2:, 2:] = self.config.initial_velocity_std ** 2 * np.eye(2)
        else:
            self.P = _covariance(covariance, 4, 'initial covariance')
        self.reference_time = self.time
        self.reference_position = pos.copy()
        self.last_innovation: np.ndarray | None = None
        self.last_innovation_covariance: np.ndarray | None = None
        self.last_gain: np.ndarray | None = None
        self.last_nis: float | None = None
        self.last_observation_time: float | None = None
        self.num_updates = 0
        self.num_reanchors = 0

    def _elapsed(self, time: float) -> float:
        time = float(time)
        if not np.isfinite(time):
            raise ValueError('Requested time must be finite')
        dt = time - self.time
        if dt < -1e-12 * max(1.0, abs(time), abs(self.time)):
            raise ValueError('A causal filter cannot move backwards in time')
        return max(0.0, dt)

    def forecast(self, time: float) -> tuple[np.ndarray, np.ndarray]:
        """Return the future mean/covariance without changing this filter."""
        dt = self._elapsed(time)
        F, Q = transition_matrices(dt, self.config.q_position, self.config.q_acceleration)
        mean = F @ self.x
        covariance = _covariance(F @ self.P @ F.T + Q, 4, 'forecast covariance')
        return (mean.copy(), covariance)

    def predict(self, dt: float) -> tuple[np.ndarray, np.ndarray]:
        """Advance the stored belief by a nonnegative elapsed time."""
        dt = float(dt)
        if not np.isfinite(dt) or dt < 0:
            raise ValueError('Prediction dt must be finite and nonnegative')
        return self.predict_to(self.time + dt)

    def predict_to(self, time: float) -> tuple[np.ndarray, np.ndarray]:
        """Advance the stored belief to an absolute time."""
        mean, covariance = self.forecast(time)
        self.x, self.P = (mean, covariance)
        self.time = max(self.time, float(time))
        return (self.x.copy(), self.P.copy())

    def reanchor(self, position, time: float) -> tuple[np.ndarray, np.ndarray]:
        """Define a new image reference at its known acquisition midpoint.

        The known actuator correction is exactly the required correction at
        that instant by definition of the new image anchor. This is a change
        of relative reference, not an oracle measurement of global drift.
        """
        position = _vector(position, 2, 'new reference correction')
        mean, covariance = self.forecast(time)
        mean[:2] = position
        new_covariance = np.zeros((4, 4))
        new_covariance[2:, 2:] = covariance[2:, 2:]
        self.x, self.P = (mean, _covariance(new_covariance, 4, 'reanchored covariance'))
        self.time = max(self.time, float(time))
        self.reference_time = self.time
        self.reference_position = position.copy()
        self.num_reanchors += 1
        return (self.x.copy(), self.P.copy())

    def observe(self, z, R, time: float | None=None) -> dict[str, Any]:
        """Assimilate a noisy measured required correction at its timestamp.

        z = applied actuator correction + measured registration mismatch.
        R may be a scalar isotropic variance or a 2x2 covariance. The Joseph
        covariance update preserves symmetry/PSD. Exact redundant observations
        are allowed, but an innovation inconsistent with zero uncertainty is
        rejected instead of being silently ignored.
        """
        z = _vector(z, 2, 'registration measurement')
        if np.ndim(R) == 0:
            R = float(R) * np.eye(2)
        R = _covariance(R, 2, 'measurement covariance')
        when = self.time if time is None else float(time)
        prior, covariance = self.forecast(when)
        H = np.zeros((2, 4))
        H[:, :2] = np.eye(2)
        innovation = z - prior[:2]
        S = _covariance(covariance[:2, :2] + R, 2, 'innovation covariance')
        try:
            K = np.linalg.solve(S, covariance[:2, :]).T
            weighted_innovation = np.linalg.solve(S, innovation)
        except np.linalg.LinAlgError:
            inverse = np.linalg.pinv(S, hermitian=True)
            projected = S @ inverse @ innovation
            if not np.allclose(projected, innovation, rtol=1e-09, atol=1e-10):
                raise ValueError('Measurement conflicts with an exactly known state')
            K = covariance[:, :2] @ inverse
            weighted_innovation = inverse @ innovation
        mean = prior + K @ innovation
        A = np.eye(4) - K @ H
        posterior = A @ covariance @ A.T + K @ R @ K.T
        self.x, self.P = (mean, _covariance(posterior, 4, 'posterior covariance'))
        self.time = max(self.time, when)
        self.last_innovation = innovation.copy()
        self.last_innovation_covariance = S.copy()
        self.last_gain = K.copy()
        self.last_nis = max(0.0, float(innovation @ weighted_innovation))
        self.last_observation_time = self.time
        self.num_updates += 1
        return {'time': self.time, 'innovation': innovation.copy(), 'innovation_covariance': S.copy(), 'gain': K.copy(), 'nis': self.last_nis, 'mean': self.x.copy(), 'covariance': self.P.copy()}

    def state_dict(self) -> dict[str, Any]:
        """JSON-compatible audit snapshot; no simulator truth is present."""
        return {'time': self.time, 'state': self.x.tolist(), 'covariance': self.P.tolist(), 'reference_time': self.reference_time, 'reference_position': self.reference_position.tolist(), 'num_updates': self.num_updates, 'num_reanchors': self.num_reanchors, 'last_observation_time': self.last_observation_time, 'last_nis': self.last_nis, 'q_position': self.config.q_position, 'q_acceleration': self.config.q_acceleration}

In [ ]:
#@title Full Kalman microscopy environment and observable state
"""Kalman forward correction for the measured-data microscopy simulator.

The controller-facing filter estimates the correction required to revisit the
material anchor of the latest image. Its state is [c_row,c_col,v_row,v_col].
IMAGE defines a new relative reference but supplies no drift-velocity evidence.
Only a paid CAL produces a noisy registration observation and a Kalman update.
SPEC can apply the causal predicted correction at its acquisition midpoint.

The registration sensor is an explicit Gaussian simulation assumption, not a
registration algorithm applied to experimental image pairs. True coordinates
are confined to the physical simulator and audit traces. The inherited finite
field, midpoint approximation, blur law, science model and return are retained.
"""
from __future__ import annotations
from dataclasses import dataclass, replace
from typing import Any
import numpy as np

@dataclass(frozen=True)
class Config(BaseConfig):
    budget: float = 100.0
    seed_pairs: int = 12
    correction_mode: str = 'kalman'
    q_position: float = 0.0256
    q_acceleration: float = 0.0025
    registration_sigma: float = 0.1
    initial_velocity_sigma: float = 0.4
    cost_randomization: bool = False
    image_cost_range: tuple[float, float] = (2.0, 4.0)
    cal_cost_range: tuple[float, float] = (0.5, 2.0)
    spectrum_duration_range: tuple[float, float] = (0.5, 2.0)

    def __post_init__(self):
        super().__post_init__()
        if self.correction_mode not in ('kalman', 'reactive'):
            raise ValueError('correction_mode must be kalman or reactive')
        if min(self.q_position, self.q_acceleration, self.registration_sigma) < 0:
            raise ValueError('Noise levels must be nonnegative')
        if self.initial_velocity_sigma <= 0:
            raise ValueError('Initial velocity uncertainty must be positive')
        for bounds in (self.image_cost_range, self.cal_cost_range, self.spectrum_duration_range):
            if len(bounds) != 2 or not 0 < bounds[0] <= bounds[1]:
                raise ValueError('Cost ranges must be positive ordered pairs')

def sample_cost_config(cfg: Config, seed: int) -> Config:
    """Draw known episode costs independently of drift/action RNG streams."""
    if not cfg.cost_randomization:
        return cfg
    rng = np.random.default_rng(int(seed) + 4300003)
    return replace(cfg, image_cost_factor=float(rng.uniform(*cfg.image_cost_range)), cal_cost=float(rng.uniform(*cfg.cal_cost_range)), spectrum_duration=float(rng.uniform(*cfg.spectrum_duration_range)), cost_randomization=False)
_TRIU = tuple(zip(*np.triu_indices(4)))
_EXTRA_NAMES = ('kf_correction_delta_row', 'kf_correction_delta_col', 'kf_velocity_row', 'kf_velocity_col') + tuple((f'kf_cov_{i}_{j}' for i, j in _TRIU)) + ('kf_spec_forward_shift_row', 'kf_spec_forward_shift_col', 'kf_spec_sigma_row', 'kf_spec_sigma_col', 'kf_innovation_row', 'kf_innovation_col', 'kf_innovation_nis', 'kf_updates_fraction', 'image_cost_factor_scaled', 'cal_cost_scaled', 'spectrum_duration_scaled', 'budget_scaled', 'kf_q_position_scaled', 'kf_q_acceleration_scaled', 'registration_sigma_scaled')

class CampaignEnv(BaseCampaignEnv):
    """Same ten choices; forward correction is automatic at SPEC execution."""
    obs_names = BaseCampaignEnv.obs_names + _EXTRA_NAMES

    def __init__(self, data: DataBundle, cfg: Config | None=None, seed: int=0, regime: str | None=None):
        self.requested_config = cfg or Config()
        self._kalman_ready = False
        super().__init__(data, sample_cost_config(self.requested_config, seed), seed, regime)

    def reset(self):
        self._kalman_ready = False
        super().reset()
        cfg = self.cfg
        self.registration_rng = np.random.default_rng(self.seed + 1700003)
        self.kf = KalmanDriftFilter(position=self.correction.copy(), velocity=np.zeros(2), time=self.t, covariance=np.diag([0.0, 0.0, cfg.initial_velocity_sigma ** 2, cfg.initial_velocity_sigma ** 2]), config=KalmanConfig(q_position=cfg.q_position, q_acceleration=cfg.q_acceleration, initial_velocity_std=cfg.initial_velocity_sigma))
        self.last_innovation = np.zeros(2)
        self.last_nis = 0.0
        self.kf_updates = 0
        self._kalman_ready = True
        return self.observation()

    def observation(self):
        base = super().observation()
        if not self._kalman_ready:
            return np.concatenate((base, np.zeros(len(_EXTRA_NAMES), dtype=np.float32)))
        cfg = self.cfg
        mean, cov = self.kf.forecast(self.t)
        future_mean, future_cov = self.kf.forecast(self.t + cfg.spectrum_duration / 2)
        scale = np.array([PATCH_SIZE, PATCH_SIZE, 1.0, 1.0])
        normalized_cov = cov / np.outer(scale, scale)
        extra = [*(mean[:2] - self.correction) / PATCH_SIZE, *mean[2:], *(normalized_cov[i, j] for i, j in _TRIU), *(future_mean[:2] - self.correction) / PATCH_SIZE, *np.sqrt(np.maximum(np.diag(future_cov)[:2], 0.0)) / PATCH_SIZE, *self.last_innovation / PATCH_SIZE, min(10.0, np.log1p(max(0.0, self.last_nis)) / np.log(101.0)), self.kf_updates / cfg.budget, cfg.image_cost_factor / 3.0, cfg.cal_cost, cfg.spectrum_duration, cfg.budget / 100.0, cfg.q_position / 0.0256, cfg.q_acceleration / 0.0025, cfg.registration_sigma / 0.1]
        result = np.concatenate((base, np.asarray(extra, np.float32)))
        if result.shape != (len(self.obs_names),) or not np.isfinite(result).all():
            raise RuntimeError('Nonfinite or inconsistent belief observation')
        return result.astype(np.float32)

    def step(self, action: int):
        action = int(action)
        if not 0 <= action < N_ACTIONS or not self.action_mask()[action]:
            raise ValueError(f'Illegal action {action} at time {self.t:.6f}')
        start = self.t
        old_correction = self.correction.copy()
        cfg = self.cfg
        prediction = None
        if action in IMAGE_ACTIONS:
            midpoint = start + cfg.image_cost_factor * cfg.taus[action] / 2
            self.kf.reanchor(position=old_correction, time=midpoint)
        elif action == SPECTRUM:
            midpoint = start + cfg.spectrum_duration / 2
            forecast, covariance = self.kf.forecast(midpoint)
            prediction = (forecast.copy(), covariance.copy())
            if cfg.correction_mode == 'kalman':
                self.correction = forecast[:2].copy()
        _, reward, done, info = super().step(action)
        event = self.trace[-1]
        if action == CALIBRATE:
            ideal_shift = self.last_measured_shift.copy()
            measured_shift = ideal_shift + self.registration_rng.normal(0.0, cfg.registration_sigma, size=2)
            observed_required_correction = old_correction + measured_shift
            mean, covariance = self.kf.forecast(self.t)
            R = np.eye(2) * cfg.registration_sigma ** 2
            innovation = observed_required_correction - mean[:2]
            innovation_cov = covariance[:2, :2] + R
            self.last_nis = float(innovation @ np.linalg.pinv(innovation_cov) @ innovation)
            self.last_innovation = innovation.copy()
            self.kf.observe(z=observed_required_correction, R=R, time=self.t)
            self.kf_updates += 1
            self.correction = observed_required_correction.copy()
            self.last_measured_shift = measured_shift.copy()
            target = self.data.centers[event['target_id']]
            event.update(measured_shift=measured_shift.tolist(), correction=self.correction.tolist(), registration_sigma=cfg.registration_sigma, registration_observed_required_correction=observed_required_correction.tolist(), kf_innovation=innovation.tolist(), kf_innovation_nis=self.last_nis, actual_position=(target + self.world.true(self.t) - self.correction).tolist())
        self.kf.predict(self.t - self.kf.time)
        event.update(correction_mode=cfg.correction_mode, correction_before=old_correction.tolist(), correction_after=self.correction.tolist(), kf_mean_after=self.kf.x.tolist(), kf_cov_after=self.kf.P.tolist(), kf_updates=self.kf_updates)
        if prediction is not None:
            forecast, covariance = prediction
            record = self.spec_trace[-1]
            position_error = np.asarray(record['spectrum_position']) - np.asarray(record['image_anchor'])
            predicted_cov = covariance[:2, :2]
            true_required = self.correction + position_error
            forecast_error = true_required - forecast[:2]
            inverse_cov = np.linalg.pinv(predicted_cov, hermitian=True)
            supported = bool(np.allclose(predicted_cov @ inverse_cov @ forecast_error, forecast_error, rtol=1e-09, atol=1e-10))
            mahalanobis = float(forecast_error @ inverse_cov @ forecast_error) if supported else None
            additions = dict(kf_forecast_midpoint=forecast.tolist(), kf_cov_midpoint=covariance.tolist(), kf_command_applied=cfg.correction_mode == 'kalman', commanded_correction=self.correction.tolist(), kf_forecast_error=forecast_error.tolist(), kf_forecast_error_norm=float(np.linalg.norm(forecast_error)), kf_forecast_rms_sigma=float(np.sqrt(np.trace(predicted_cov))), kf_forecast_mahalanobis2=mahalanobis, kf_forecast_ellipse95_covered=bool(supported and mahalanobis <= 5.9914645471), kf_forecast_degenerate_support_failure=not supported)
            record.update(additions)
            event.update(additions)
        info['audit'] = event.copy()
        return (self.observation(), reward, done, info)

    def metrics(self) -> dict[str, Any]:
        result = super().metrics()

        def average(key):
            values = [x[key] for x in self.spec_trace if key in x]
            return float(np.mean(values)) if values else float('nan')
        result.update(correction_mode=self.cfg.correction_mode, image_cost_factor=self.cfg.image_cost_factor, cal_cost=self.cfg.cal_cost, spectrum_duration=self.cfg.spectrum_duration, registration_sigma=self.cfg.registration_sigma, mean_kf_forecast_error=average('kf_forecast_error_norm'), mean_kf_forecast_rms_sigma=average('kf_forecast_rms_sigma'), kf_forecast_ellipse95_coverage=average('kf_forecast_ellipse95_covered'), kf_measurement_updates=self.kf_updates, observation_dim=len(self.obs_names), filter_reference='required correction relative to latest image anchor; velocity persists', registration_sensor='Gaussian simulated relative registration; measured only by CAL')
        return result

## Load EELS data with `gdown`, then calculate PCA from raw spectra
This cell downloads `Plasmonic_sets_hackathon.npy` from the original Google Drive ID, verifies SHA-256, extracts the 33×63×4322 cube and structural image when present, partitions measured locations, and fits PCA on calibration-fit spectra only. It does **not** read pretrained model files. If the archive format is different from expected, it fails explicitly rather than silently using synthetic data.

In [ ]:
#@title Download raw EELS with gdown, preprocess PCA targets from scratch
import subprocess
try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown
from sklearn.decomposition import PCA

RAW_DRIVE_ID = '1FKmlRTRDImQS91V4YQjcQHoJU19X6P_6'
RAW_FILENAME = 'Plasmonic_sets_hackathon.npy'
RAW_SHA256 = '5f571cc1fda0ba551a335a8d332582973c0a56632a12bb1de7a35bff92e3ff1e'
DATASET_KEY = '2'
RAW_PATH = Path(RAW_FILENAME)

if not RAW_PATH.is_file() or RAW_PATH.stat().st_size != 584089774:
    print('Downloading the original EELS file (~584 MB) using gdown ...')
    result = gdown.download(id=RAW_DRIVE_ID, output=str(RAW_PATH), quiet=False, fuzzy=True)
    if not result or not RAW_PATH.exists():
        raise RuntimeError('gdown could not download the public EELS file. Check Drive sharing/access or Google Drive quota.')

def sha256_file(filename):
    h=hashlib.sha256()
    with open(filename,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
actual_sha=sha256_file(RAW_PATH)
if actual_sha != RAW_SHA256:
    raise ValueError(f'Raw data hash mismatch: {actual_sha}; expected {RAW_SHA256}. Do not silently substitute a different file.')
print('Verified raw data SHA-256:',actual_sha)

# Inspect arrays recursively without assuming that the original numpy archive
# is a flat array. Object .npy containers and nested dicts are supported.
def array_candidates(obj, prefix='root', depth=0, seen=None):
    if seen is None:seen=set()
    if depth>7 or id(obj) in seen:return
    seen.add(id(obj))
    if isinstance(obj,np.ndarray):
        if obj.dtype==object:
            if obj.ndim==0:
                yield from array_candidates(obj.item(),prefix+'.item',depth+1,seen)
            elif obj.size<=32:
                for k,v in np.ndenumerate(obj):
                    yield from array_candidates(v,prefix+str(k),depth+1,seen)
        elif obj.ndim in (2,3):
            yield prefix,obj
    elif isinstance(obj,dict):
        for k,v in obj.items():
            yield from array_candidates(v,prefix+'.'+str(k),depth+1,seen)
    elif isinstance(obj,(tuple,list)):
        for k,v in enumerate(obj):
            yield from array_candidates(v,prefix+'.'+str(k),depth+1,seen)

raw = np.load(RAW_PATH, allow_pickle=True, mmap_mode=None)
if isinstance(raw,np.ndarray) and raw.dtype==object and raw.ndim==0:
    raw=raw.item()
# The original dataset identifies acquisition 2, but the archive may encode it
# as int or string. Restrict extraction to this subset where possible.
chosen=raw
if isinstance(raw,dict):
    for key in (DATASET_KEY,int(DATASET_KEY)):
        if key in raw:
            chosen=raw[key]
            break
candidates=list(array_candidates(chosen))
print('Array candidates:',[(name,tuple(arr.shape),str(arr.dtype)) for name,arr in candidates[:24]])

cube_candidates=[]
for name,arr in candidates:
    if arr.ndim!=3:continue
    sh=arr.shape
    if 33 in sh and 63 in sh and max(sh)>=1000:
        cube_candidates.append((name,arr))
if len(cube_candidates)!=1:
    raise ValueError('Expected one EELS cube with spatial dimensions 33x63 and >=1000 channels; found '+repr([(k,a.shape) for k,a in cube_candidates]))
cube_name,cube=cube_candidates[0]
# Convert only the axis order, not an expensive duplicate of the full cube.
spatial_axes=(list(cube.shape).index(33),list(cube.shape).index(63))
energy_axis=next(i for i in range(3) if i not in spatial_axes)
cube=np.transpose(cube,(*spatial_axes,energy_axis))
if cube.shape[:2]!=(33,63):raise ValueError(f'Unexpected spatial orientation: {cube.shape}')

image_candidates=[]
for name,arr in candidates:
    if arr.ndim==2 and arr.shape in ((33,63),(63,33)):
        a=arr if arr.shape==(33,63) else arr.T
        image_candidates.append((name,a))
if image_candidates:
    image_name,image_raw=image_candidates[0]
    image=np.asarray(image_raw,dtype=np.float32)
    print('Structural image source:',image_name)
else:
    # A reproducible structural proxy when the .npy archive contains only
    # spectral datacubes; this is *not* identical to an independent HAADF image.
    image=np.asarray(np.nanmean(cube,axis=-1),dtype=np.float32)
    print('WARNING: no separate 2D structural image; using mean EELS intensity proxy.')
image=np.nan_to_num(image,nan=float(np.nanmedian(image)),posinf=0,neginf=0)

# The 100-point interpolated representation and five PCA components are learned
# anew from the CAL_FIT partition; the held-out test spectra never enter PCA.
flat=cube.reshape((-1,cube.shape[-1]))
valid=np.isfinite(flat).all(axis=1) & (np.nanstd(flat,axis=1)>1e-12)
valid_ids=np.flatnonzero(valid)
if len(valid_ids)<1500:
    raise ValueError(f'Too few finite measured spectra: {len(valid_ids)}')
print('Raw cube',cube.shape,'valid spectra',len(valid_ids))

split_rng=np.random.default_rng(42)
ids=split_rng.permutation(valid_ids)
# Stable fractional split scaled to the observed valid-spectra count; for the
# previously used file N=2079 and the counts are 333/83/1247/416.
n=len(ids)
n_fit=round(n*.16); n_ref=round(n*.04); n_test=round(n*.20)
seed_ids=np.sort(ids[:n_fit]); ref_ids=np.sort(ids[n_fit:n_fit+n_ref]); test_ids=np.sort(ids[n_fit+n_ref:n_fit+n_ref+n_test]); pool_ids=np.sort(ids[n_fit+n_ref+n_test:])

energy_grid=np.linspace(0,cube.shape[-1]-1,100)
lo=np.floor(energy_grid).astype(int); hi=np.ceil(energy_grid).astype(int)
w=(energy_grid-lo).astype(np.float32)
def project_raw_spectra(raw_ids):
    spectra=np.asarray(flat[np.asarray(raw_ids,dtype=np.int64)],dtype=np.float32)
    return spectra[:,lo]*(1-w)[None,:]+spectra[:,hi]*w[None,:]
train_100=project_raw_spectra(seed_ids)
# Center/scale solely on CAL_FIT data.
channel_mean=train_100.mean(axis=0)
channel_scale=train_100.std(axis=0)
channel_scale=np.where(channel_scale>1e-8,channel_scale,1.)
train_normalized=(train_100-channel_mean)/channel_scale
pca=PCA(n_components=5,random_state=42,svd_solver='full').fit(train_normalized)
pca_scale=np.maximum(np.std(pca.transform(train_normalized),axis=0),1e-8)
def targets(raw_ids):
    x=(project_raw_spectra(raw_ids)-channel_mean)/channel_scale
    return (pca.transform(x)/pca_scale).astype(np.float32)
metadata={'source_drive_id': RAW_DRIVE_ID,'source_filename':RAW_FILENAME,
          'source_sha256':actual_sha,'dataset_key':DATASET_KEY,
          'cube_shape':list(map(int,cube.shape)),
          'structural_image_source':image_candidates[0][0] if image_candidates else 'EELS integrated-intensity proxy',
          'split_seed':42,'split_counts':{'cal_fit':len(seed_ids),'reference':len(ref_ids),'pool':len(pool_ids),'hidden_test':len(test_ids)},
          'target':'five PCA components from 100 spectral points; PCA and scaling fitted only to CAL_FIT',
          'heldout_isolation':'test excluded from fitting, seed samples and acquisition lookup',
          'real_data':True,'synthetic_fallback':False}
DATA=DataBundle(image=image,pool_ids=pool_ids,seed_ids=seed_ids,
                ref_ids=ref_ids,test_ids=test_ids,
                pool_targets=targets(pool_ids),seed_targets=targets(seed_ids),
                hidden_test_targets=targets(test_ids),metadata=metadata)
print('Prepared data:',DATA.image.shape,metadata['split_counts'])
# Free the large source cube; prepared experimental data remain in DATA.
del flat,cube,raw,candidates,train_100,train_normalized
import gc;gc.collect()


## RUDDER and PPO — fully in-notebook learning
Collect structured warmups, train the recurrent predictor on complete returns, redistribute credits with a terminal residual, and update the stochastic recurrent PPO policy.

In [ ]:
#@title Full recurrent policy, return predictor, redistribution and PPO
"""Causal RUDDER reward redistribution and matched recurrent policy baselines.

This copy uses the adjacent Kalman-aware environment through ordinary imports.
Policy and return-predictor input dimensions are discovered from observations.

The scientific objective is an undiscounted, fixed-budget terminal return G.
RUDDER predicts G after each causal prefix and redistributes the realized G by
prediction differences plus a terminal residual. No simulator-only metrics or
positions enter either network. With lambda=1 and full return-to-go targets,
redistribution is mathematically a learned history-dependent baseline; it is
not a guarantee of better policy learning than an already competent critic.
"""
from __future__ import annotations
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence
import copy
import json
import time
import numpy as np
import torch
from torch import nn

@dataclass
class LearningConfig:
    episodes: int = 128
    batch_size: int = 8
    warmup_episodes: int = 32
    hidden_size: int = 48
    return_hidden_size: int = 64
    ppo_epochs: int = 4
    ppo_clip: float = 0.2
    policy_lr: float = 0.0003
    return_lr: float = 0.001
    return_epochs: int = 12
    initial_return_epochs: int = 30
    return_batch_size: int = 24
    entropy_coef: float = 0.02
    value_coef: float = 0.4
    max_grad_norm: float = 0.5
    gae_lambda: float = 1.0
    replay_capacity: int = 384
    torch_threads: int = 1
    reward_scale: float | None = None
    regimes: tuple[str, ...] = ('random', 'linear', 'periodic')

@dataclass
class Episode:
    """Observable trajectory and separately stored simulator audit metadata."""
    observations: np.ndarray
    next_observations: np.ndarray
    actions: np.ndarray
    masks: np.ndarray
    old_log_probs: np.ndarray
    old_values: np.ndarray
    direct_rewards: np.ndarray
    terminal_score: float
    seed: int
    regime: str
    metrics: dict[str, Any] = field(default_factory=dict)
    trace: list[dict[str, Any]] = field(default_factory=list)
    redistributed_rewards: np.ndarray | None = None
    prefix_predictions: np.ndarray | None = None
    terminal_residual: float | None = None

    def __len__(self) -> int:
        return len(self.actions)

def make_policy_inputs(episode: Episode) -> np.ndarray:
    """Observation at decision time plus the previous chosen action."""
    previous = np.zeros((len(episode), episode.masks.shape[1]), np.float32)
    if len(episode) > 1:
        previous[np.arange(1, len(episode)), episode.actions[:-1]] = 1.0
    return np.concatenate((episode.observations, previous), axis=1).astype(np.float32)

def make_return_inputs(episode: Episode) -> np.ndarray:
    """Causal action outcome: pre/post observable state and selected action.

    G, direct reward, oracle drift, realized positions and test error are not
    features. The online learner's uncertainty may be part of observations.
    """
    actions = np.eye(episode.masks.shape[1], dtype=np.float32)[episode.actions]
    return np.concatenate((episode.observations, actions, episode.next_observations), axis=1).astype(np.float32)

class RecurrentActorCritic(nn.Module):

    def __init__(self, observation_dim: int, action_dim: int=10, hidden_size: int=48):
        super().__init__()
        self.observation_dim = observation_dim
        self.action_dim = action_dim
        self.hidden_size = hidden_size
        self.encoder = nn.Sequential(nn.Linear(observation_dim + action_dim, hidden_size), nn.Tanh())
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.actor = nn.Linear(hidden_size, action_dim)
        self.critic = nn.Linear(hidden_size, 1)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        with torch.no_grad():
            self.actor.bias.copy_(torch.log(torch.tensor([0.4 / 8] * 8 + [0.2, 0.4])))

    def forward(self, inputs: torch.Tensor, hidden: torch.Tensor | None=None):
        features, hidden = self.gru(self.encoder(inputs), hidden)
        return (self.actor(features), self.critic(features).squeeze(-1), hidden)

class ReturnPredictor(nn.Module):
    """Unidirectional sequence model: output k sees only inputs 0...k."""

    def __init__(self, input_dim: int, hidden_size: int=64):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_size = hidden_size
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_size), nn.Tanh())
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
        self.register_buffer('target_mean', torch.tensor(0.0))
        self.register_buffer('target_scale', torch.tensor(1.0))

    def forward(self, inputs: torch.Tensor, lengths=None) -> torch.Tensor:
        features, _ = self.gru(self.encoder(inputs))
        normalized = self.head(features).squeeze(-1)
        return self.target_mean + self.target_scale * normalized

def redistribute_rewards(predictions: Sequence[float], terminal_score: float):
    """Return prediction differences and exact last-step residual in raw units.

    g_0 is defined as zero. The final correction is G - g_K. Consequently
    sum(rewards)==G (to floating-point precision), including negative rewards.
    """
    values = np.asarray(predictions, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or (not np.isfinite(values).all()):
        raise ValueError('Expected a nonempty finite vector of prefix predictions')
    if not np.isfinite(terminal_score):
        raise ValueError('The terminal score must be finite')
    rewards = np.diff(np.concatenate(([0.0], values)))
    residual = float(terminal_score - values[-1])
    rewards[-1] += residual
    rewards[-1] += float(terminal_score - rewards.sum(dtype=np.float64))
    return (rewards, residual)

def _environment(data, env_config, seed, regime):
    return CampaignEnv(data, env_config, seed=seed, regime=regime)

def _observation(value) -> np.ndarray:
    result = np.asarray(value, dtype=np.float32).reshape(-1)
    if not np.isfinite(result).all():
        raise ValueError('Nonfinite policy observation; repair environment normalization')
    return result

def workflow_selector(calibration_probability: float=0.5, image_index: int | None=None):
    """Simple observational baseline or diverse warmup behavior.

    Calibration is chosen before spectroscopy with the specified probability.
    A completed calibration is followed by a spectrum whenever affordable.
    """

    def choose(observation, mask, rng, prior_actions):
        images = np.flatnonzero(mask[:8])
        if mask[9]:
            if prior_actions and prior_actions[-1] == 8:
                return 9
            if mask[8] and rng.random() < calibration_probability:
                return 8
            return 9
        if len(images):
            if image_index is not None and mask[image_index]:
                return int(image_index)
            return int(rng.choice(images))
        return int(rng.choice(np.flatnonzero(mask)))
    return choose

def collect_episode(policy: RecurrentActorCritic | None, data, env_config, seed: int, regime: str='random', deterministic: bool=False, action_selector: Callable | None=None) -> Episode:
    """Run one complete physical-time-budget campaign.

    action_selector, when provided, sees observations/masks/past actions only.
    Its episodes are appropriate for warmup/diagnostics, not on-policy PPO.
    """
    env = _environment(data, env_config, seed, regime)
    obs = _observation(env.reset())
    rng = np.random.default_rng(seed + 901)
    generator = torch.Generator(device='cpu').manual_seed(seed + 902)
    hidden = None
    previous_action = np.zeros(10, np.float32)
    observations, next_observations, actions, masks = ([], [], [], [])
    log_probs, values, deltas = ([], [], [])
    done = False
    if policy is not None:
        policy.eval()
    while not done:
        mask = np.asarray(env.action_mask(), dtype=bool)
        if mask.shape != (10,) or not mask.any():
            if actions:
                break
            raise RuntimeError('No feasible action at campaign start')
        log_probability, value = (0.0, 0.0)
        distribution = None
        if policy is not None:
            inp = torch.from_numpy(np.concatenate((obs, previous_action)))[None, None]
            with torch.no_grad():
                logits, predicted_values, hidden = policy(inp, hidden)
                logits = logits[0, 0].masked_fill(~torch.from_numpy(mask), -1000000000.0)
                distribution = torch.distributions.Categorical(logits=logits)
                value = float(predicted_values[0, 0])
        if action_selector is not None:
            action = int(action_selector(obs.copy(), mask.copy(), rng, actions))
        elif distribution is not None:
            action = int(logits.argmax()) if deterministic else int(torch.multinomial(distribution.probs, 1, generator=generator))
        else:
            action = int(rng.choice(np.flatnonzero(mask)))
        if not mask[action]:
            raise RuntimeError(f'Controller chose infeasible action {action}')
        if distribution is not None:
            log_probability = float(distribution.log_prob(torch.tensor(action)))
        next_obs, delta, done, info = env.step(action)
        next_obs = _observation(next_obs)
        observations.append(obs)
        next_observations.append(next_obs)
        actions.append(action)
        masks.append(mask)
        log_probs.append(log_probability)
        values.append(value)
        deltas.append(float(delta))
        previous_action = np.eye(10, dtype=np.float32)[action]
        obs = next_obs
        if len(actions) > 10000:
            raise RuntimeError('Episode exceeded 10000 positive-duration actions')
    score = env.terminal_score
    if callable(score):
        score = score()
    return Episode(observations=np.asarray(observations, np.float32), next_observations=np.asarray(next_observations, np.float32), actions=np.asarray(actions, np.int64), masks=np.asarray(masks, bool), old_log_probs=np.asarray(log_probs, np.float32), old_values=np.asarray(values, np.float32), direct_rewards=np.asarray(deltas, np.float64), terminal_score=float(score), seed=int(seed), regime=str(regime), metrics=copy.deepcopy(env.metrics()), trace=copy.deepcopy(env.trace))

def collect_warmup(data, env_config, seed: int, config: LearningConfig) -> list[Episode]:
    """Share diverse campaigns across reward methods within one environment.

    Correction modes use matched campaign seeds and schedules; their actual
    observed outcomes differ and are therefore collected separately.
    """
    episodes = []
    for i in range(config.warmup_episodes):
        regime = config.regimes[i % len(config.regimes)]
        calibration_probability = (0.0, 0.25, 0.75, 1.0)[i % 4]
        image_index = i % 8 if i % 3 else None
        episodes.append(collect_episode(None, data, env_config, seed=100000 + 10000 * seed + i, regime=regime, action_selector=workflow_selector(calibration_probability, image_index)))
    return episodes

def _pad_sequences(sequences, dtype=np.float32):
    lengths = np.asarray([len(s) for s in sequences], dtype=np.int64)
    shape = (len(sequences), int(lengths.max()), *np.asarray(sequences[0]).shape[1:])
    padded = np.zeros(shape, dtype=dtype)
    valid = np.zeros(shape[:2], dtype=bool)
    for i, sequence in enumerate(sequences):
        padded[i, :lengths[i]] = sequence
        valid[i, :lengths[i]] = True
    return (torch.from_numpy(padded), torch.from_numpy(valid), lengths)

def predictor_diagnostics(predictor: ReturnPredictor, episodes: Sequence[Episode]):
    """Unweighted campaign errors; each prefix fraction is scored separately."""
    if not episodes:
        return {}
    predictor.eval()
    predictions, targets = ([], np.asarray([e.terminal_score for e in episodes]))
    with torch.no_grad():
        for ep in episodes:
            p = predictor(torch.from_numpy(make_return_inputs(ep))[None])[0].numpy().astype(float)
            predictions.append(p)
    output: dict[str, Any] = {'episodes': len(episodes)}
    denominator = float(np.mean((targets - float(predictor.target_mean)) ** 2))
    for fraction, name in ((0.25, 'quarter'), (0.5, 'half'), (0.75, 'three_quarters'), (1.0, 'final')):
        selected = np.asarray([p[min(len(p) - 1, max(0, int(np.ceil(fraction * len(p))) - 1))] for p in predictions])
        mse = float(np.mean((selected - targets) ** 2))
        output[f'{name}_rmse'] = float(np.sqrt(mse))
        output[f'{name}_skill_vs_warmup_mean'] = float(1 - mse / denominator) if denominator > 1e-16 else None
    output['mean_abs_final_residual'] = float(np.mean([abs(e.terminal_score - p[-1]) for e, p in zip(episodes, predictions)]))
    return output

def fit_return_model(predictor: ReturnPredictor, train_episodes: Sequence[Episode], validation_episodes: Sequence[Episode]=(), *, epochs: int=12, batch_size: int=24, learning_rate: float=0.001, optimizer=None, seed: int=0):
    """Fit terminal return at every prefix, with whole-episode validation.

    Early stopping selects only among this call's epochs. Callers keep the
    validation campaign identities out of training across every update.
    """
    if not train_episodes:
        return {'epochs': 0}
    optimizer = optimizer or torch.optim.Adam(predictor.parameters(), lr=learning_rate)
    rng = np.random.default_rng(seed)
    best_state, best_validation, best_epoch = (None, float('inf'), 0)
    losses = []
    scale = max(float(predictor.target_scale), 1e-08)
    for epoch in range(epochs):
        predictor.train()
        permutation = rng.permutation(len(train_episodes))
        epoch_losses = []
        for start in range(0, len(permutation), batch_size):
            batch = [train_episodes[i] for i in permutation[start:start + batch_size]]
            inputs, valid, lengths = _pad_sequences([make_return_inputs(ep) for ep in batch])
            target = torch.tensor([ep.terminal_score for ep in batch], dtype=torch.float32)[:, None]
            predicted = predictor(inputs)
            errors = ((predicted - target) / scale) ** 2
            prefix_loss = ((errors * valid).sum(1) / valid.sum(1)).mean()
            terminal_loss = errors[torch.arange(len(batch)), torch.tensor(lengths - 1)].mean()
            loss = prefix_loss + 0.25 * terminal_loss
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(predictor.parameters(), 1.0)
            optimizer.step()
            epoch_losses.append(float(loss.detach()))
        losses.append(float(np.mean(epoch_losses)))
        if validation_episodes:
            diagnostics = predictor_diagnostics(predictor, validation_episodes)
            validation = float(np.mean([diagnostics[f'{name}_rmse'] ** 2 for name in ('quarter', 'half', 'three_quarters', 'final')]))
        else:
            validation = losses[-1]
        if validation < best_validation:
            best_validation, best_epoch = (validation, epoch + 1)
            best_state = copy.deepcopy(predictor.state_dict())
    if best_state is not None:
        predictor.load_state_dict(best_state)
    predictor.eval()
    return {'epochs': int(epochs), 'best_epoch': best_epoch, 'normalized_training_loss': losses[-1], 'validation': predictor_diagnostics(predictor, validation_episodes)}

def predict_and_redistribute(predictor: ReturnPredictor, episode: Episode):
    predictor.eval()
    with torch.no_grad():
        predictions = predictor(torch.from_numpy(make_return_inputs(episode))[None])[0].numpy().astype(np.float64)
    rewards, residual = redistribute_rewards(predictions, episode.terminal_score)
    episode.prefix_predictions = predictions
    episode.redistributed_rewards = rewards
    episode.terminal_residual = residual
    return rewards

def _training_rewards(method, ep, predictor):
    if method == 'terminal':
        rewards = np.zeros(len(ep), np.float64)
        rewards[-1] = ep.terminal_score
        return rewards
    if method == 'direct':
        rewards = ep.direct_rewards.copy()
        if not np.isclose(rewards.sum(), ep.terminal_score, rtol=1e-06, atol=1e-09):
            raise RuntimeError('Direct uncertainty changes do not sum to the terminal objective')
        return rewards
    if method == 'rudder':
        return predict_and_redistribute(predictor, ep)
    raise ValueError('method must be terminal, direct, or rudder')

def _ppo_update(policy, optimizer, episodes, rewards, config, reward_scale):
    inputs, valid, _ = _pad_sequences([make_policy_inputs(ep) for ep in episodes])
    masks, _, _ = _pad_sequences([ep.masks for ep in episodes], dtype=bool)
    masks[~valid, 0] = True
    actions, _, _ = _pad_sequences([ep.actions for ep in episodes], dtype=np.int64)
    old_log_probs, _, _ = _pad_sequences([ep.old_log_probs for ep in episodes])
    advantages, value_targets = ([], [])
    for ep, raw_rewards in zip(episodes, rewards):
        r = np.asarray(raw_rewards, np.float64) / reward_scale
        adv = np.zeros(len(ep), np.float64)
        accumulator = 0.0
        for t in reversed(range(len(ep))):
            next_value = float(ep.old_values[t + 1]) if t + 1 < len(ep) else 0.0
            delta = r[t] + next_value - float(ep.old_values[t])
            accumulator = delta + config.gae_lambda * accumulator
            adv[t] = accumulator
        advantages.append(adv.astype(np.float32))
        value_targets.append((adv + ep.old_values).astype(np.float32))
    advantage, _, _ = _pad_sequences(advantages)
    target, _, _ = _pad_sequences(value_targets)
    selected = advantage[valid]
    advantage = (advantage - selected.mean()) / (selected.std(unbiased=False) + 1e-08)
    records = []
    policy.train()
    for _ in range(config.ppo_epochs):
        logits, values, _ = policy(inputs)
        logits = logits.masked_fill(~masks, -1000000000.0)
        distribution = torch.distributions.Categorical(logits=logits)
        log_prob = distribution.log_prob(actions)
        ratio = (log_prob - old_log_probs).exp()
        unclipped = ratio * advantage
        clipped = ratio.clamp(1 - config.ppo_clip, 1 + config.ppo_clip) * advantage
        policy_loss = -torch.minimum(unclipped, clipped)[valid].mean()
        value_loss = 0.5 * ((values - target) ** 2)[valid].mean()
        entropy = distribution.entropy()[valid].mean()
        loss = policy_loss + config.value_coef * value_loss - config.entropy_coef * entropy
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), config.max_grad_norm)
        optimizer.step()
        records.append({'policy_loss': float(policy_loss.detach()), 'value_loss': float(value_loss.detach()), 'entropy': float(entropy.detach()), 'approx_kl': float((old_log_probs - log_prob)[valid].mean().detach()), 'clip_fraction': float(((ratio - 1).abs() > config.ppo_clip)[valid].float().mean())})
    policy.eval()
    return {key: float(np.mean([record[key] for record in records])) for key in records[0]}

def train_controller(data, env_config, method: str, seed: int=0, config: LearningConfig | None=None, output_dir: str | Path | None=None, warmup_episodes: Sequence[Episode] | None=None, progress: Callable[[dict], None] | None=None):
    """Train matched recurrent controllers and optionally save portable weights.

    Predictor fitting precedes collection of each new policy batch. The frozen
    predictor labels that batch before any of its episodes enter replay.
    Every fifth replay episode is permanently reserved for validation.
    """
    config = config or LearningConfig()
    if method not in ('terminal', 'direct', 'rudder'):
        raise ValueError('Unknown reward method')
    if not 0 <= config.gae_lambda <= 1 or config.episodes <= 0:
        raise ValueError('Invalid training configuration')
    torch.set_num_threads(config.torch_threads)
    torch.manual_seed(seed)
    start_time = time.perf_counter()
    warmup = list(warmup_episodes) if warmup_episodes is not None else collect_warmup(data, env_config, seed, config)
    if not warmup:
        raise ValueError('At least one common warmup episode is required')
    observation_dim = warmup[0].observations.shape[1]
    scores = np.asarray([ep.terminal_score for ep in warmup], dtype=float)
    normalization_scores = scores[[i % 5 != 0 for i in range(len(warmup))]]
    if not len(normalization_scores):
        normalization_scores = scores
    reward_scale = config.reward_scale or max(float(normalization_scores.std()), float(np.median(np.abs(normalization_scores))) * 0.1, 0.0001)
    torch.manual_seed(seed)
    policy = RecurrentActorCritic(observation_dim, 10, config.hidden_size)
    optimizer = torch.optim.Adam(policy.parameters(), lr=config.policy_lr)
    predictor = None
    predictor_optimizer = None
    replay_train = [ep for i, ep in enumerate(warmup) if i % 5 != 0]
    replay_validation = [ep for i, ep in enumerate(warmup) if i % 5 == 0]
    if not replay_train:
        replay_train, replay_validation = (warmup, [])
    if method == 'rudder':
        torch.manual_seed(seed + 50000)
        predictor = ReturnPredictor(2 * observation_dim + 10, config.return_hidden_size)
        predictor.target_mean.fill_(float(normalization_scores.mean()))
        predictor.target_scale.fill_(float(reward_scale))
        predictor_optimizer = torch.optim.Adam(predictor.parameters(), lr=config.return_lr)
    history, completed, replay_index = ([], [], len(warmup))
    for batch_index, batch_start in enumerate(range(0, config.episodes, config.batch_size)):
        fit_diagnostics = {}
        if predictor is not None:
            fit_diagnostics = fit_return_model(predictor, replay_train, replay_validation, epochs=config.initial_return_epochs if batch_index == 0 else config.return_epochs, batch_size=config.return_batch_size, learning_rate=config.return_lr, optimizer=predictor_optimizer, seed=seed + batch_index)
        batch = []
        for index in range(batch_start, min(batch_start + config.batch_size, config.episodes)):
            regime = config.regimes[index % len(config.regimes)]
            batch.append(collect_episode(policy, data, env_config, seed=200000 + 10000 * seed + index, regime=regime, deterministic=False))
        prospective = predictor_diagnostics(predictor, batch) if predictor is not None else {}
        rewards = [_training_rewards(method, ep, predictor) for ep in batch]
        optimization = _ppo_update(policy, optimizer, batch, rewards, config, reward_scale)
        record = {'method': method, 'training_seed': seed, 'batch': batch_index, 'episodes': batch_start + len(batch), 'mean_return': float(np.mean([ep.terminal_score for ep in batch])), 'std_return': float(np.std([ep.terminal_score for ep in batch])), 'mean_actions': float(np.mean([len(ep) for ep in batch])), 'mean_images': float(np.mean([np.count_nonzero(ep.actions < 8) for ep in batch])), 'mean_calibrations': float(np.mean([np.count_nonzero(ep.actions == 8) for ep in batch])), 'mean_spectrum_attempts': float(np.mean([np.count_nonzero(ep.actions == 9) for ep in batch])), 'image_action_counts': [int(sum((np.count_nonzero(ep.actions == action) for ep in batch))) for action in range(8)], 'max_reward_sum_error': float(max((abs(float(r.sum()) - ep.terminal_score) for r, ep in zip(rewards, batch)))), 'elapsed_seconds': time.perf_counter() - start_time, **optimization, 'predictor_fit': fit_diagnostics, 'prospective_prediction': prospective}
        history.append(record)
        completed.extend(batch)
        for ep in batch:
            (replay_validation if replay_index % 5 == 0 else replay_train).append(ep)
            replay_index += 1
        replay_train = replay_train[-config.replay_capacity:]
        replay_validation = replay_validation[-max(8, config.replay_capacity // 4):]
        if progress is not None:
            progress(record)
    result = {'method': method, 'seed': seed, 'policy': policy, 'predictor': predictor, 'history': history, 'episodes': completed, 'warmup_episodes': warmup, 'reward_scale': float(reward_scale), 'config': asdict(config), 'training_seconds': time.perf_counter() - start_time, 'predictor_validation': predictor_diagnostics(predictor, replay_validation) if predictor is not None else {}}
    return result

def evaluate_controller(policy: RecurrentActorCritic | None, data, env_config, seeds: Iterable[int], regimes: Sequence[str]=('random', 'linear', 'periodic'), deterministic: bool=True, predictor: ReturnPredictor | None=None, action_selector: Callable | None=None):
    """Return audit rows and complete causal traces on unseen campaign seeds."""
    seeds = list(seeds)
    episodes, rows = ([], [])
    for regime in regimes:
        for seed in seeds:
            ep = collect_episode(policy, data, env_config, seed, regime, deterministic, action_selector)
            if predictor is not None:
                predict_and_redistribute(predictor, ep)
            row = {'seed': int(seed), 'regime': regime, 'terminal_score': ep.terminal_score, 'actions': len(ep), **ep.metrics}
            rows.append(row)
            episodes.append(ep)
    numeric_keys = [key for key, value in rows[0].items() if isinstance(value, (int, float, np.number)) and key != 'seed'] if rows else []
    summary = {key: {'mean': float(np.mean([row[key] for row in rows])), 'std': float(np.std([row[key] for row in rows]))} for key in numeric_keys}
    return {'rows': rows, 'episodes': episodes, 'summary': summary, 'predictor_diagnostics': predictor_diagnostics(predictor, episodes) if predictor is not None else {}}

## Train from scratch
Default: 64 warmups + 384 policy-training campaigns per seed, 3 independent seeds. For a quick local/Colab smoke test, use 4 warmups, 8 policy campaigns, and one training seed.

In [ ]:
#@title Train RUDDER + Kalman RUDDER and PPO from random initialization
ENVIRONMENT_CONFIG = Config(budget=100.0, correction_mode="kalman", image_cost_factor=3.0, cal_cost=1.0, spectrum_duration=1.0, registration_sigma=0.10, q_position=0.0256, q_acceleration=0.0025, velocity=0.40, random_walk_scale=0.16, periodic_amplitude=3.0, drift_period=48.0, cost_randomization=COST_RANDOMIZATION)
CONDITION_LABEL='RUDDER + Kalman'
assert TRAINING_EPISODES % POLICY_BATCH_SIZE == 0
assert RUN_TRAINING, 'This notebook intentionally trains from scratch. Set RUN_TRAINING=True.'
MODELS,TRAINING_HISTORY,TRAINING_CHECKS={},[],[]
for seed in TRAINING_SEEDS:
    lc=LearningConfig(episodes=TRAINING_EPISODES,warmup_episodes=WARMUP_EPISODES,batch_size=POLICY_BATCH_SIZE,replay_capacity=max(384,TRAINING_EPISODES))
    warmups=collect_warmup(DATA,ENVIRONMENT_CONFIG,seed,lc)
    gs=np.asarray([ep.terminal_score for ep in warmups],dtype=float)
    scale=max(float(gs.std()),float(np.median(np.abs(gs)))*.1,1e-4)
    lc=replace(lc,reward_scale=scale)
    def progress(row,seed=seed):
        if row['batch']%max(1,(TRAINING_EPISODES//POLICY_BATCH_SIZE)//8)==0 or row['episodes']==TRAINING_EPISODES:
            print(f'{CONDITION_LABEL} | seed {seed} | {row["episodes"]}/{TRAINING_EPISODES} campaigns | G={row["mean_return"]:.5f}',flush=True)
    fitted=train_controller(DATA,ENVIRONMENT_CONFIG,'rudder',seed=seed,config=lc,warmup_episodes=warmups,progress=progress)
    MODELS[seed]={'policy':fitted['policy'],'predictor':fitted['predictor']}
    for row in fitted['history']:
        TRAINING_HISTORY.append({'condition':CONDITION_LABEL,'seed':seed,**{k:v for k,v in row.items() if k!='image_action_counts' and not isinstance(v,dict)}})
    TRAINING_CHECKS.append({'seed':seed,'max_credit_conservation_error':max(x['max_reward_sum_error'] for x in fitted['history']),'predictor_validation':fitted['predictor_validation']})
TRAINING_FRAME=pd.DataFrame(TRAINING_HISTORY)
display(TRAINING_FRAME.tail())


In [ ]:
#@title Basic invariants: partitions, reward accounting, and physical behavior
assert not set(DATA.pool_ids)&set(DATA.test_ids)
assert DATA.pool_targets.shape[1]==5
assert abs(sum([.2,.3142857142857143])-.5142857142857143)<1e-8
env=CampaignEnv(DATA,ENVIRONMENT_CONFIG,seed=97531,regime='linear')
ob=env.reset()
assert env.action_mask().shape==(10,)
assert np.isfinite(ob).all()
print('Basic causal/environment checks passed; observation features:',len(ob))


## Evaluate trained policy on unseen campaign seeds
Report terminal return, scientific test MSE, spectrum positioning error, calibration usage, and valid spectra separately for each drift regime.

In [ ]:
#@title Evaluate independently held-out campaigns
EVALUATION_ROWS=[]
EVALUATION_EPISODES={}
for seed in TRAINING_SEEDS:
    model=MODELS[seed]
    result=evaluate_controller(model['policy'],DATA,ENVIRONMENT_CONFIG,range(EVALUATION_FIRST_SEED,EVALUATION_FIRST_SEED+EVALUATION_SEEDS_PER_REGIME),regimes=DRIFT_REGIMES,deterministic=False,predictor=model['predictor'])
    EVALUATION_EPISODES[(CONDITION_LABEL,seed)]=result['episodes']
    for row in result['rows']:
        EVALUATION_ROWS.append({'condition':CONDITION_LABEL,'training_seed':seed,**row})
RESULTS=pd.DataFrame(EVALUATION_ROWS)
print('Completed held-out campaigns:',len(RESULTS))
display(RESULTS.groupby('regime')[[k for k in ('terminal_score','heldout_mse','mean_pair_mismatch','n_calibrations','n_successful_pairs') if k in RESULTS]].agg(['mean','std']).round(5))
assert max(abs(float(ep.redistributed_rewards.sum())-ep.terminal_score) for eps in EVALUATION_EPISODES.values() for ep in eps)<1e-5


In [ ]:
#@title Training, scientific performance, and positioning results
fig,axs=plt.subplots(2,2,figsize=(12,8))
piv=TRAINING_FRAME.pivot(index='episodes',columns='seed',values='mean_return')
x=piv.index.to_numpy();mu=piv.mean(axis=1).to_numpy();sd=piv.std(axis=1).fillna(0).to_numpy()
axs[0,0].plot(x,mu);axs[0,0].fill_between(x,mu-sd,mu+sd,alpha=.2)
axs[0,0].set(xlabel='Policy-training campaigns per seed',ylabel='Mean terminal reward',title='RUDDER policy training')
for ax,metric,title in [(axs[0,1],'terminal_score','Final reward'),(axs[1,0],'heldout_mse','Spectral test MSE'),(axs[1,1],'mean_pair_mismatch','Position mismatch, px')]:
    vals=RESULTS.groupby(['regime','training_seed'])[metric].mean().groupby('regime')
    means=vals.mean(); errors=vals.std().fillna(0)
    ax.bar(np.arange(len(means)),means.values,yerr=errors.values,capsize=5)
    ax.set_xticks(range(len(means)),means.index);ax.set_title(title)
for ax in axs.flat:ax.grid(alpha=.2)
fig.tight_layout();plt.show()


## Visualize learned credits
The heatmap uses **realized actions**, so unobserved action/time bins remain blank. Calibration and spectroscopy credits are displayed separately.

In [ ]:
#@title RUDDER credits by action and campaign time; image acquisition heatmap
records=[]
for (label,seed),eps in EVALUATION_EPISODES.items():
    for ep in eps:
        for j,action in enumerate(ep.actions):
            elapsed=ENVIRONMENT_CONFIG.budget*(1-float(ep.observations[j,0]))
            records.append({'condition':label,'training_seed':seed,'regime':ep.regime,'elapsed':elapsed,'action':int(action),'credit':float(ep.redistributed_rewards[j]),'boundary':j==0 or j==len(ep.actions)-1})
CREDIT_FRAME=pd.DataFrame(records)
d=CREDIT_FRAME[~CREDIT_FRAME.boundary]
fig,axs=plt.subplots(1,2,figsize=(14,5))
for a,name in ((8,'CALIBRATE'),(9,'SPECTRUM')):
    p=d[d.action==a];axs[0].scatter(p.elapsed,p.credit,s=12,alpha=.4,label=f'{name} (n={len(p)})')
axs[0].axhline(0,color='gray',lw=.7);axs[0].set(xlabel='Elapsed campaign time',ylabel='RUDDER credit',title='Executed calibration and spectrum credits');axs[0].legend()
bins=np.linspace(45,100,17);m=np.full((8,len(bins)-1),np.nan)
p=d[d.action<8].copy();p['timebin']=np.clip(np.digitize(p.elapsed,bins)-1,0,len(bins)-2)
for (setting,tbin),group in p.groupby(['action','timebin']):m[int(setting),int(tbin)]=group.credit.mean()
cmap=plt.get_cmap('coolwarm').copy();cmap.set_bad('#dddddd')
scale=max(float(np.nanpercentile(abs(m),95)) if np.isfinite(m).any() else 0,1e-5)
im=axs[1].imshow(m,origin='lower',aspect='auto',extent=(bins[0],bins[-1],-.5,7.5),cmap=cmap,vmin=-scale,vmax=scale)
axs[1].set_yticks(range(8),[f'{blur_sigma(t):.2f}' for t in ENVIRONMENT_CONFIG.taus]);axs[1].set(xlabel='Elapsed campaign time',ylabel='Image blur σ (pixels)',title='Mean observed IMAGE credits (unseen bins blank)')
fig.colorbar(im,ax=axs[1],label='RUDDER credit');fig.tight_layout();plt.show()


In [ ]:
#@title Inspect one held-out campaign, return forecasts and credits
ep=EVALUATION_EPISODES[(CONDITION_LABEL,TRAINING_SEEDS[0])][0]
times=[ENVIRONMENT_CONFIG.budget*(1-float(o[0])) for o in ep.observations]
fig,ax=plt.subplots(2,1,figsize=(12,6),sharex=True)
ax[0].step(times,ep.actions,where='post');ax[0].set(ylabel='Action index',title=CONDITION_LABEL+' — held-out campaign')
ax[1].plot(times,ep.prefix_predictions,label='Predicted final return')
ax[1].scatter(times,ep.redistributed_rewards,s=16,label='Redistributed credit')
ax[1].axhline(ep.terminal_score,color='gray',ls='--',label='Realized final return');ax[1].legend()
ax[1].set(xlabel='Elapsed time',ylabel='Return / credit')
fig.tight_layout();plt.show()


In [ ]:
#@title Spectrum-midpoint correction error and Kalman uncertainty in one held-out campaign
example=EVALUATION_EPISODES[(CONDITION_LABEL,TRAINING_SEEDS[0])][0]
measurements=[e for e in example.trace if e.get('kind')=='SPECTRUM' and 'kf_forecast_error_norm' in e]
calibrations=[e for e in example.trace if e.get('kind')=='CALIBRATE']
fig,axes=plt.subplots(1,2,figsize=(13,4))
if measurements:
    t=np.array([e['time'] for e in measurements],float)
    err=np.array([e['kf_forecast_error_norm'] for e in measurements],float)
    sig=np.array([e['kf_forecast_rms_sigma'] for e in measurements],float)
    axes[0].plot(t,err,'o-',label='True midpoint correction error')
    axes[0].plot(t,sig,'--',label='Predicted 2D RMS σ')
    axes[0].legend(fontsize=8)
for e in calibrations:
    for a in axes:a.axvline(e['time'],color='gray',alpha=.13)
axes[0].set(xlabel='Elapsed time',ylabel='Position error / uncertainty (px)',title='Forward correction versus uncertainty')
axes[1].scatter(t,err/(sig+1e-8),s=18)
axes[1].axhline(1,color='gray',lw=.8)
axes[1].set(xlabel='Elapsed time',ylabel='Error / predicted RMS σ',title='Calibration of forward prediction')
fig.tight_layout();plt.show()


In [ ]:
#@title Export newly generated results (no saved results used as inputs)
RESULTS.to_csv('RUDDER_Kalman_heldout.csv',index=False)
TRAINING_FRAME.to_csv('RUDDER_Kalman_training.csv',index=False)
CREDIT_FRAME.to_csv('RUDDER_Kalman_credits.csv',index=False)
print('CSV outputs saved to the current Colab working directory.')
